In [ ]:
'''
Run with the following to be able to import from 'core_experiments' & `remoteobj` natively
`uv run --no-project --python 3.12 --with risq/software/arm --with risq/software/common/remoteobj/ --with ipython ipython`

Alternatevly use whatever environment manager you prefer
'''

In [1]:
'''
The following demonstrates how to connect to several boards with remoteobj
'''
# UNDEFINED = "UNDEFINED"
from os import getenv; from pathlib import Path; from remoteobj.client import Proxy
# RISQ_HOST, RISQ_PORTS = getenv("RISQ_HOST", UNDEFINED), [int(port) for port in getenv("RISQ_PORTS", UNDEFINED).split(",")]
# RISQ_ROOT, RISQ_PROJECT =Path(getenv("RISQ_ROOT", UNDEFINED)),"risq"

RISQ_HOST, RISQ_PROJECT = "192.168.0.13", "risq"
session = Proxy.to_host(host=RISQ_HOST)
RISQS = [session.RISQ.__call__]

# if RISQ_HOST != UNDEFINED:
#     if RISQ_PORTS == [UNDEFINED]:
#         sessions = [Proxy.to_host(host=RISQ_HOST)]
#     else:
#         sessions = [Proxy.to_host(host=RISQ_HOST, port=port) for port in RISQ_PORTS]
#     RISQS = [session.RISQ.__call__ for session in sessions]
# else:
#     raise RuntimeError(
#         "No target board defined to attach to." \
#         "Please define a RISQ_HOST in your environment with the IP of your board."
#         )
risqs = [RISQ(project=RISQ_PROJECT,
              xsa=Path("risq", "build", "risq.xsa"),
              dtbo=Path("risq", "build", "risq.dtbo"))
              for RISQ in RISQS]

risq/build/risq.dtbo: 100%|██████████| 1/1 [00:00<00:00, 181.07kB/s]


In [ ]:
'''
For this demo I'll just need a single board
'''
from core_experiments import CoreExperiments
experiment = CoreExperiments(risqs[0])
experiment.run_orc(log2_shots=10, phase_offset=0, graph=False)
experiment.run_t1(rounds=100, time_steps=500, time_increment_in_cycles=500, shots=10)
experiment.run_rabi(rounds=100, initial_gain=-25000, gain_steps=500, gain_increment=100, shots=10)
experiment.run_ramsey(rounds=100, time_steps=500, time_increment_in_cycles=125, phase_step=0, shots=10)

In [2]:
from dwave.elm.interface.pulse_definitions import PulseDefinitions
from pprint import pprint
pulse_definitions = PulseDefinitions()

system_names = [str(system.system.name) for system in pulse_definitions.elm_environment_state.system_versions]
print(f"{system_names=}\n\n")
print("-"*20)

for system_name in system_names:
    system = pulse_definitions.get_system_by_name(pulse_definitions.elm_environment_state, system_name)
    print(f"{system=}\n")

    parameter_roles = pulse_definitions.role_labels(pulse_definitions.registry, system)
    print(f"{parameter_roles=}\n")

    if parameters := pulse_definitions.get_parameters_by_role(system, "pulse_definitions"):
        all_pulse_data = pulse_definitions.get_pulse_data(system)
        print(f"{all_pulse_data=}\n")

        for pulse_name, pulse_data in all_pulse_data.items():
            pulse_parameters = pulse_data["parameters"]
            print(f"Pulse: {pulse_name}\n  Parameters: {pulse_parameters=}")

        quantum_frequency = pulse_definitions.get_quantum_frequency(system)
        print(f"{quantum_frequency=}\n")

    card = pulse_definitions.get_system_by_name(
        environment_state=pulse_definitions.elm_environment_state, name=system.name
    )

    if (mixer_freq_dac0 := pulse_definitions.get_mixer_freq_dac0(system)) != None:
        print(f"{mixer_freq_dac0=}")
    print("-"*20)


system_names=['q0.mem_readout', 'card1_q0.pair1', 'q0.mem_transmon', 'card1_q0.configuration']


--------------------
system=System(uuid=UUID('1cbd578b-818b-4c5f-893d-c05c7886e7f9'), name='q0.mem_readout', kind_ref=SystemKindRef(uuid=UUID('2de8ea74-395c-4897-b6a0-e3e7abf244d5'), version_number=1), committed_at=None, attributes=[EntityAttribute(role_name='mode_id', values=[IntValue(value_type=<ValueType.INT: 'int'>, int_value=3)])], parameters=[EntityParameter(role_name='quantum_frequency', values=[FloatValue(value_type=<ValueType.FLOAT: 'float'>, float_value=1500000000.0)]), EntityParameter(role_name='aithex_ssb', values=[FloatValue(value_type=<ValueType.FLOAT: 'float'>, float_value=50000000.0)]), EntityParameter(role_name='log_true_acquisition_length', values=[IntValue(value_type=<ValueType.INT: 'int'>, int_value=2500)]), EntityParameter(role_name='log_true_acquisition_delay', values=[IntValue(value_type=<ValueType.INT: 'int'>, int_value=300)]), EntityParameter(role_name='pulse_defini

In [7]:
risqs[0].rfdc.dac_tiles[0].blocks[0].MixerSettings['Freq'] = 0.01
print(f"DAC0 Freq: {risqs[0].rfdc.dac_tiles[0].blocks[0].MixerSettings['Freq']}")

system = pulse_definitions.get_system_by_name(pulse_definitions.elm_environment_state, 'card1_q0.configuration')
if freq := pulse_definitions.get_mixer_freq_dac0(system):
    risqs[0].rfdc.dac_tiles[0].blocks[0].MixerSettings['Freq'] = freq*(1E6)
    print(f"DAC0 Freq: {risqs[0].rfdc.dac_tiles[0].blocks[0].MixerSettings['Freq']}")

risqs[0].rfdc.dac_tiles[0].blocks[0].MixerSettings['Freq'] = 0
print(f"DAC0 Freq: {risqs[0].rfdc.dac_tiles[0].blocks[0].MixerSettings['Freq']}")

DAC0 Freq: 0.00999999999606871
DAC0 Freq: 4125000000.0
DAC0 Freq: 0.0
